In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# global data definitions
DATASETS_PATH = '/home/sgeraci/Desktop/datasets/CICIDS2017/MachineLearningCVE'

MONDAY = f'{DATASETS_PATH}/Monday-WorkingHours.pcap_ISCX.csv'
TUESDAY = f'{DATASETS_PATH}/Tuesday-WorkingHours.pcap_ISCX.csv'
WEDNESDAY = f'{DATASETS_PATH}/Wednesday-workingHours.pcap_ISCX.csv'
THURSDAY1 =  f'{DATASETS_PATH}/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
THURSDAY2 =  f'{DATASETS_PATH}/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'
FRIDAY1 = f'{DATASETS_PATH}/Friday-WorkingHours-Morning.pcap_ISCX.csv'
FRIDAY2 = f'{DATASETS_PATH}/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
FRIDAY3 = f'{DATASETS_PATH}/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv'

In [ ]:
# read datasets
mon = pd.read_csv(MONDAY)
thu = pd.read_csv(TUESDAY)
wed = pd.read_csv(WEDNESDAY)
thu1 = pd.read_csv(THURSDAY1)
thu2 = pd.read_csv(THURSDAY2)
fri1 = pd.read_csv(FRIDAY1)
fri2 = pd.read_csv(FRIDAY2)
fri3 = pd.read_csv(FRIDAY3)

print(f"File: {MONDAY} has {mon.shape[0]} rows and {mon.shape[1]} columns")
print(f"File: {TUESDAY} has {thu.shape[0]} rows and {thu.shape[1]} columns")
print(f"File: {WEDNESDAY} has {wed.shape[0]} rows and {wed.shape[1]} columns")
print(f"File: {THURSDAY1} has {thu1.shape[0]} rows and {thu1.shape[1]} columns")
print(f"File: {THURSDAY2} has {thu2.shape[0]} rows and {thu2.shape[1]} columns")
print(f"File: {FRIDAY1} has {fri1.shape[0]} rows and {fri1.shape[1]} columns")
print(f"File: {FRIDAY2} has {fri2.shape[0]} rows and {fri2.shape[1]} columns")
print(f"File: {FRIDAY3} has {fri3.shape[0]} rows and {fri3.shape[1]} columns")

In [ ]:
weekdays = pd.DataFrame(pd.concat([mon, thu, wed, thu1, thu2, fri1, fri2, fri3], ignore_index=True))
weekdays.head()

In [ ]:
# remove NaN and ±Inf values
weekdays.replace([np.inf, -np.inf], np.nan, inplace=True)
print(f"Row with at least one NaN in weekdays dataset: {weekdays[weekdays.isna().any(axis=1)].shape[0]}")
weekdays.dropna(inplace=True)
print(f"✔ NaN values removed, new shape: {weekdays.shape}")

In [ ]:
# remove columns with only 0s
zero_columns = weekdays.columns[(weekdays == 0).all()]
if not zero_columns.empty:
    print(f"Columns with only 0s: {zero_columns.tolist()}")
    weekdays.drop(columns=zero_columns, inplace=True)
    print(f"✔ Columns with only 0s removed, new shape: {weekdays.shape}")

In [ ]:
### just rename Labels that contain non-printable characters 
print("Before...")
print(weekdays.loc[:," Label"].unique())

weekdays.loc[:," Label"].replace({"Web Attack � XSS" : "XSS", "Web Attack � Sql Injection": "Sql Injection", "Web Attack � Brute Force": "Brute Force"}, inplace=True)
print("After..")
print(weekdays.loc[:," Label"].unique())

## remove trailing && leading spaces from all the labels
rename_cols = lambda col_lbl: col_lbl.strip()
weekdays.rename(rename_cols, axis=1, inplace=True, errors="raise")

In [ ]:
weekdays['Label'] = (weekdays['Label'] != 'BENIGN').astype(int)  # Convert 'BENIGN' to 1 and others to 0
print('New values in Label column:', weekdays['Label'].unique())

In [ ]:
# Plotting the number of benign and attack samples
label_counts = weekdays['Label'].value_counts().sort_index()
colors = ['skyblue', 'salmon']  # Benign: skyblue, Attack: salmon
plt.bar(['Benign', 'Attack'], label_counts, color=colors)
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Number of Benign and Attack Samples')
plt.xticks([0, 1], ['Benign', 'Attack'])
# plt.yticks(range(0, int(label_counts.max())+1, max(1, int(label_counts.max()/10))))
plt.show()

In [ ]:
# Check if the 'Label' column exists and has the same number of rows as the rest of the DataFrame
labels = weekdays['Label']
weekdays = weekdays.drop(columns=['Label'])
if labels.shape[0] == weekdays.shape[0]:
    print("Labels and weekdays rows match.")
else:
    print("Warning: Labels and weekdays rows do not match!")

In [ ]:
# X_tr, X_val, Y_tr, Y_val = train_test_split(weekdays, labels, random_state=42)
# feature_names = X_tr.columns
# forest = RandomForestClassifier(random_state=0)
# forest.fit(X_tr, Y_tr)
# res = permutation_importance(forest, X_val, Y_val, n_repeats=10, random_state=42, n_jobs=2)
# forest_importances = pd.Series(res.importances_mean, index=feature_names).sort_values(ascending=False)

In [ ]:
# # Plotting feature importances
# fig, ax = plt.subplots(figsize=(8,6))
# forest_importances.plot.bar(yerr=res.importances_std, ax=ax)
# ax.set_title("Feature importances using permutation on full model")
# ax.set_ylabel("Mean accuracy decrease")
# fig.tight_layout()
# print(forest_importances.keys()[:20])
# plt.show()